# 01 — INGESTION LAYER
### Influenza A · NCBI · dua jalur ingestion · seluruh arsip

**Peta ke laporan:** Bab 2.1 Metode Pengumpulan Data · LO 2 · Sesi 3 *Ingestion Layer*


| | Jalur A | Jalur B |
|---|---|---|
| **Pola ingestion** | *Bulk extract* berkas besar | *Multisource extractor* terstruktur |
| **Protokol** | NCBI E-utilities (`esearch` + `efetch`) | NCBI Virus `vvsearch2` (Solr) |
| **Format** | FASTA — **tak terstruktur** | CSV — **terstruktur** |
| **Isi** | Sekuens nukleotida + defline | Subtipe, segmen, inang, negara, tanggal |
| **Peran** | Bahan mentah fitur *k*-mer | **Label** untuk klasifikasi + dimensi geografi |

Keduanya saling melengkapi dan **tidak bisa saling menggantikan**: FASTA tidak membawa label subtipe yang bisa dipercaya, sedangkan CSV tidak membawa sekuens. Penggabungannya lewat nomor aksesi dilakukan di notebook 02, dan selisih antar keduanya menjadi metrik kualitas data.

**Tidak ada penyaringan tanggal.** Seluruh arsip diambil, dari rekaman terlama sampai rilis terbaru.

---
**Keluaran notebook ini**
- `D:\BDA\lake\fasta\` — berkas `.fasta.gz` per partisi tahun
- `D:\BDA\lake\meta_csv\` — berkas `.csv` per partisi tahun
- `D:\BDA\lake\manifest_ingestion.json` — bukti audit: jumlah, ukuran, durasi



## Bootstrap

In [1]:
import sys, subprocess
sys.path.insert(0, r"D:\BDA\nb")

# paket yang belum ada di environment tf_gpu
for p in ["pymysql", "sqlalchemy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)

from bda_common import *

info_mesin()
print()
try:
    RAHASIA = muat_secret()
    print(f"Kredensial dibaca. API key NCBI: "
          f"{'AKTIF (10 permintaan/detik)' if CFG['ncbi_api_key'] else 'tidak dipakai (3 permintaan/detik)'}")
except FileNotFoundError as e:
    print(e)
    RAHASIA = None

CPU logis      : 24
RAM total      : 50.5 GB   bebas 44.7 GB
Disk D: bebas  : 201.4 GB dari 1,024.1 GB
Python         : 3.10.12
Mode           : KLASTER  (hdfs://namenode:8020)
run_id         : run_20260922T020600Z

Kredensial dibaca. API key NCBI: AKTIF (10 permintaan/detik)


## LAPISAN 1 — DATA SOURCES

In [2]:
import urllib.parse
import pandas as pd

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
DATASETS = "https://api.ncbi.nlm.nih.gov/datasets/v2alpha"
VVSEARCH = "https://www.ncbi.nlm.nih.gov/genomes/VirusVariation/vvsearch2/"

HTTP = sesi_http()


def _params_ncbi(p):
    if CFG["ncbi_api_key"]:
        p = dict(p, api_key=CFG["ncbi_api_key"])
    return p


def ambil(url, params=None, timeout=None, percobaan=None, stream=False):
    percobaan = percobaan or CFG["http_retry"]
    for i in range(percobaan):
        try:
            r = HTTP.get(url, params=params, timeout=timeout or CFG["http_timeout"],
                         stream=stream)
            r.raise_for_status()
            time.sleep(CFG["ncbi_delay"])
            return r
        except Exception as ex:
            tunggu = min(2 ** i, 60)
            print(f"      retry {i+1}/{percobaan} {type(ex).__name__} — tunggu {tunggu}s")
            time.sleep(tunggu)
    raise RuntimeError(f"gagal setelah {percobaan} percobaan: {url}")


def json_aman(teks):
    try:
        return json.loads(teks, strict=False)
    except json.JSONDecodeError:
        return json.loads(re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", " ", teks),
                          strict=False)


def hitung_entrez(term):
    r = ambil(EUTILS + "/esearch.fcgi", _params_ncbi(
        {"db": "nuccore", "term": term, "retmode": "json", "retmax": 0}))
    return int(json_aman(r.text)["esearchresult"]["count"])


TERM_DASAR = f'txid{CFG["taxid"]}[Organism]'

with Tahap("probe kedua sumber", "DATA SOURCES"):
    n_entrez = hitung_entrez(TERM_DASAR)
    print(f"  Jalur A — Entrez nuccore      : {n_entrez:,} sekuens")

    r = ambil(f"{DATASETS}/virus/taxon/{CFG['taxid']}/dataset_report",
              {"page_size": 1})
    n_datasets = json_aman(r.text)["total_count"]
    print(f"  Pembanding — NCBI Datasets v2 : {n_datasets:,} sekuens")

    selisih = abs(n_entrez - n_datasets)
    print(f"\n  Selisih antar sumber: {selisih:,} "
          f"({100*selisih/max(n_entrez,1):.3f}%) — wajar, keduanya "
          f"mengindeks pada waktu yang sedikit berbeda.")


--------------------------------------------------------------------
[>] DATA SOURCES | probe kedua sumber
  Jalur A — Entrez nuccore      : 1,662,259 sekuens
  Pembanding — NCBI Datasets v2 : 1,654,585 sekuens

  Selisih antar sumber: 7,674 (0.462%) — wajar, keduanya mengindeks pada waktu yang sedikit berbeda.
[<] OK | 4.7 detik | RAM bebas 44.7 GB


### Rencana partisi

Menarik 1,6 juta rekaman dalam satu permintaan tidak realistis, dan penomoran halaman yang dalam (*deep paging*) sangat lambat di kedua sumber. Solusinya: **partisi menurut tahun rilis GenBank**.

Setiap partisi selalu diambil dari posisi awal, sehingga tidak pernah ada *offset* dalam. Efek sampingnya menguntungkan: partisi menjadi unit *checkpoint* yang alami, dan berkas keluarannya langsung siap dibaca paralel oleh Spark.

In [3]:
TAHUN_AWAL, TAHUN_AKHIR = 1980, datetime.now(timezone.utc).year

def term_tahun(y):
    return f"{TERM_DASAR} AND {y}/01/01:{y}/12/31[PDAT]"

with Tahap("inventaris per tahun rilis", "DATA SOURCES"):
    # rekaman sebelum TAHUN_AWAL disatukan dalam satu partisi
    inv = [{"partisi": f"pre{TAHUN_AWAL}",
            "term": f"{TERM_DASAR} AND 1900/01/01:{TAHUN_AWAL-1}/12/31[PDAT]"}]
    for y in range(TAHUN_AWAL, TAHUN_AKHIR + 1):
        inv.append({"partisi": str(y), "term": term_tahun(y)})

    for baris in inv:
        baris["jumlah"] = hitung_entrez(baris["term"])

    df_inv = pd.DataFrame(inv)
    df_inv = df_inv[df_inv["jumlah"] > 0].reset_index(drop=True)
    total_partisi = int(df_inv["jumlah"].sum())

    print(f"  {len(df_inv)} partisi berisi data")
    print(f"  Total terhitung : {total_partisi:,}")
    print(f"  Total esearch   : {n_entrez:,}")
    print(f"  Cocok           : {'YA' if total_partisi == n_entrez else 'ADA SELISIH'}")
    print(f"\n  Partisi terbesar:")
    for _, b in df_inv.nlargest(6, "jumlah").iterrows():
        print(f"    {b['partisi']:>8}  {b['jumlah']:>9,}")

    df_inv.to_csv(LAKE / "inventaris_partisi.csv", index=False)
    print(f"\n  Estimasi permintaan efetch: "
          f"{total_partisi // CFG['efetch_batch'] + len(df_inv):,}")


--------------------------------------------------------------------
[>] DATA SOURCES | inventaris per tahun rilis
  42 partisi berisi data
  Total terhitung : 1,662,259
  Total esearch   : 1,662,259
  Cocok           : YA

  Partisi terbesar:
        2025    286,373
        2026    202,313
        2024    190,943
        2023    111,009
        2019    101,769
        2022     74,036

  Estimasi permintaan efetch: 3,366
[<] OK | 47.2 detik | RAM bebas 44.6 GB


### Temuan untuk Bab 3.4 — `[Segment]` bukan field yang sah

Pendekatan yang tampak wajar adalah memisahkan unduhan per segmen dengan query `4[Segment]`. **Itu tidak bekerja.** Basis data `nuccore` tidak punya field `Segment`; NCBI tidak menolak query tersebut, melainkan diam-diam menafsirkannya sebagai pencarian teks bebas.

Sel di bawah membuktikannya secara kuantitatif. Simpan hasilnya untuk laporan: ini contoh nyata kegagalan senyap yang hanya ketahuan kalau angkanya diperiksa. Konsekuensinya untuk desain pipeline: **nomor segmen tidak boleh diambil dari query, melainkan harus datang dari metadata jalur B dan dari defline FASTA.**

In [4]:
with Tahap("bukti: [Segment] bukan field yang sah", "DATA QUALITY"):
    r = ambil(EUTILS + "/einfo.fcgi", _params_ncbi({"db": "nuccore", "retmode": "json"}))
    fields = {f["name"] for f in json_aman(r.text)["einforesult"]["dbinfo"][0]["fieldlist"]}
    print(f"  Field 'SEGM' ada di nuccore? {'SEGM' in fields}")
    print(f"  Field yang tersedia: {len(fields)} — tidak satupun untuk segmen\n")

    total_semu = 0
    for s in range(1, 9):
        n = hitung_entrez(f"{TERM_DASAR} AND {s}[Segment]")
        total_semu += n
        print(f"    {s}[Segment] -> {n:>9,}")
    print(f"\n  Jumlah kedelapan 'segmen' : {total_semu:,}")
    print(f"  Padahal seluruh arsip     : {n_entrez:,}")
    print(f"  Kelebihan                 : {total_semu - n_entrez:,} "
          f"({total_semu/max(n_entrez,1):.2f}x lipat)")
    print("\n  Kesimpulan: query itu mencacah rekaman yang sama berkali-kali.")
    print("  Nomor segmen diambil dari jalur B dan defline, bukan dari query.")


--------------------------------------------------------------------
[>] DATA QUALITY | bukti: [Segment] bukan field yang sah
  Field 'SEGM' ada di nuccore? False
  Field yang tersedia: 34 — tidak satupun untuk segmen

    1[Segment] ->   878,784
    2[Segment] ->   738,563
    3[Segment] ->   285,814
    4[Segment] ->   368,265
    5[Segment] ->   381,444
    6[Segment] ->   292,566
    7[Segment] ->   287,492
    8[Segment] ->   234,277

  Jumlah kedelapan 'segmen' : 3,467,205
  Padahal seluruh arsip     : 1,662,259
  Kelebihan                 : 1,804,946 (2.09x lipat)

  Kesimpulan: query itu mencacah rekaman yang sama berkali-kali.
  Nomor segmen diambil dari jalur B dan defline, bukan dari query.
[<] OK | 8.3 detik | RAM bebas 44.5 GB


---
## LAPISAN 2 — JALUR A · Bulk extract sekuens (FASTA)

`esearch` dengan riwayat server (`usehistory=y`) untuk mengunci hasil pencarian satu partisi, lalu `efetch` menariknya per 500 rekaman.

Tiga hal yang membuatnya tahan gangguan:

- **Partisi ditulis utuh atau tidak sama sekali.** Berkas ditulis ke `.part` lalu diganti nama setelah lengkap, sehingga tidak pernah ada `.fasta.gz` yang separuh jadi.
- **Riwayat yang kedaluwarsa diperbarui otomatis.** `WebEnv` bisa hangus di tengah unduhan panjang; bila itu terjadi, `esearch` dijalankan ulang untuk partisi tersebut.
- **Partisi besar dipecah per bulan** agar `retstart` tidak pernah terlalu dalam.

In [5]:
AMBANG_PECAH = 60_000


def esearch_riwayat(term):
    r = ambil(EUTILS + "/esearch.fcgi", _params_ncbi(
        {"db": "nuccore", "term": term, "usehistory": "y",
         "retmode": "json", "retmax": 0}))
    js = json_aman(r.text)["esearchresult"]
    return int(js["count"]), js["webenv"], js["querykey"]


def unduh_partisi_fasta(nama, term, jumlah):
    final = LAKE_FASTA / f"flua_{nama}.fasta.gz"
    if final.exists():
        return 0, True

    sementara = LAKE_FASTA / f"flua_{nama}.part"
    total, webenv, qkey = esearch_riwayat(term)
    if total == 0:
        final.touch()
        return 0, True

    B = CFG["efetch_batch"]
    ditulis, offset, t0 = 0, 0, time.time()

    with gzip.open(sementara, "wt", encoding="utf-8") as fo:
        while offset < total:
            try:
                r = ambil(EUTILS + "/efetch.fcgi", _params_ncbi(
                    {"db": "nuccore", "query_key": qkey, "WebEnv": webenv,
                     "retstart": offset, "retmax": B,
                     "rettype": "fasta", "retmode": "text"}))
                teks = r.text
            except RuntimeError:
                # kemungkinan riwayat sudah hangus — ambil ulang lalu lanjut
                print("      riwayat diperbarui")
                total, webenv, qkey = esearch_riwayat(term)
                continue

            if "<ERROR>" in teks or not teks.strip():
                total_baru, webenv, qkey = esearch_riwayat(term)
                total = total_baru
                continue

            fo.write(teks if teks.endswith("\n") else teks + "\n")
            ditulis += teks.count(">")
            offset += B

            if offset % (B * 20) == 0:
                laju = ditulis / max(time.time() - t0, 1)
                print(f"      {min(offset,total):,}/{total:,} "
                      f"({100*min(offset,total)/total:.0f}%) — {laju:,.0f} sekuens/detik")

    sementara.replace(final)
    return ditulis, False


def rencana_partisi(df):
    tugas = []
    for _, b in df.iterrows():
        nama, term, n = b["partisi"], b["term"], int(b["jumlah"])
        if n <= AMBANG_PECAH or not nama.isdigit():
            tugas.append((nama, term, n))
        else:
            y = int(nama)
            for m in range(1, 13):
                akhir = [31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31][m - 1]
                tugas.append((
                    f"{y}m{m:02d}",
                    f"{TERM_DASAR} AND {y}/{m:02d}/01:{y}/{m:02d}/{akhir}[PDAT]",
                    None))
    return tugas

In [6]:
# ═══ JALANKAN JALUR A ═══
# Aman dihentikan kapan saja (Interrupt Kernel). Jalankan ulang untuk melanjutkan:
# partisi yang sudah punya berkas .fasta.gz akan dilewati.

TUGAS_A = rencana_partisi(df_inv)
print(f"{len(TUGAS_A)} partisi dalam antrean\n")

with Tahap("jalur A — unduh FASTA seluruh arsip", "INGESTION"):
    total_baru, dilewati, t0 = 0, 0, time.time()
    for i, (nama, term, n) in enumerate(TUGAS_A, 1):
        perkiraan = f"~{n:,}" if n else "?"
        print(f"  [{i}/{len(TUGAS_A)}] {nama} ({perkiraan})")
        ditulis, sudah_ada = unduh_partisi_fasta(nama, term, n)
        if sudah_ada:
            dilewati += 1
            print("      sudah ada — dilewati")
        else:
            total_baru += ditulis
            berlalu = time.time() - t0
            sisa = len(TUGAS_A) - i
            print(f"      {ditulis:,} sekuens | total sesi ini {total_baru:,} | "
                  f"sisa {sisa} partisi ~{sisa * berlalu / max(i - dilewati, 1) / 60:.0f} menit")

    print(f"\n  Baru diunduh : {total_baru:,}")
    print(f"  Dilewati     : {dilewati} partisi")
    print(f"  Ukuran zona  : {ukuran(LAKE_FASTA)}")

152 partisi dalam antrean


--------------------------------------------------------------------
[>] INGESTION | jalur A — unduh FASTA seluruh arsip
  [1/152] 1982 (~17)
      sudah ada — dilewati
  [2/152] 1983 (~10)
      sudah ada — dilewati
  [3/152] 1985 (~7)
      sudah ada — dilewati
  [4/152] 1988 (~3)
      sudah ada — dilewati
  [5/152] 1989 (~12)
      sudah ada — dilewati
  [6/152] 1990 (~5)
      sudah ada — dilewati
  [7/152] 1991 (~15)
      sudah ada — dilewati
  [8/152] 1992 (~5)
      sudah ada — dilewati
  [9/152] 1993 (~877)
      sudah ada — dilewati
  [10/152] 1994 (~169)
      sudah ada — dilewati
  [11/152] 1995 (~108)
      sudah ada — dilewati
  [12/152] 1996 (~135)
      sudah ada — dilewati
  [13/152] 1997 (~381)
      sudah ada — dilewati
  [14/152] 1998 (~251)
      sudah ada — dilewati
  [15/152] 1999 (~580)
      sudah ada — dilewati
  [16/152] 2000 (~651)
      sudah ada — dilewati
  [17/152] 2001 (~1,079)
      sudah ada — dilewati
  [18/152] 2002 (~1,

---
## LAPISAN 2 — JALUR B · Multisource extractor metadata (CSV)

Endpoint `vvsearch2` adalah yang dipakai situs NCBI Virus sendiri. Hanya jalur ini yang menyediakan kolom **`Serotype_s`** — dan subtipe itulah label untuk klasifikasi di notebook 05.

Dua perilaku endpoint ini yang ditemukan lewat pengujian dan harus diakali:

- **Parameter `sort` menyebabkan galat HTTP 500.** Jangan dipakai.
- **Parameter `rows` dan `start` sama-sama diabaikan.** Saya mengujinya: permintaan dengan `start=550000` dan `start=1100000` mengembalikan berkas yang identik, sama-sama dimulai dari rekaman pertama. Artinya penomoran halaman tidak mungkin dilakukan di endpoint ini, dan **partisi lewat `fq` adalah satu-satunya cara memotong hasil**.

In [7]:
KOLOM_VV = ["AccVer_s", "Serotype_s", "Segment_s", "Host_s", "CountryFull_s",
            "Region_s", "CollectionDate_s", "SLen_i", "Completeness_s",
            "Isolate_s", "CreateDate_dt", "Definition_s"]

FQ_DASAR = ['{!tag=SeqType_s}SeqType_s:("Nucleotide")',
            f'VirusLineageId_ss:({CFG["taxid"]})']


def url_vvsearch(fq_tambahan=None):
    bagian = [("q", "*:*")]
    for fq in FQ_DASAR + (fq_tambahan or []):
        bagian.append(("fq", fq))
    bagian += [("cmd", "download"), ("dlfmt", "csv"),
               ("fl", ",".join(KOLOM_VV)), ("start", "0")]
    return VVSEARCH + "?" + urllib.parse.urlencode(bagian)


def csv_utuh(p):
    import csv as _csv
    if not p.exists() or p.stat().st_size == 0:
        return False, 0, "berkas kosong"
    with open(p, "rb") as f:
        f.seek(-1, 2)
        if f.read(1) not in (b"\n", b"\r"):
            return False, 0, "baris terakhir terpotong"
    with open(p, encoding="utf-8", errors="replace", newline="") as f:
        r = _csv.reader(f)
        try:
            header = next(r)
        except StopIteration:
            return False, 0, "tanpa header"
        if "AccVer_s" not in header:
            return False, 0, "header tidak dikenali"
        n = 0
        for baris in r:
            if len(baris) != len(header):
                return False, n, f"kolom tidak konsisten di baris {n + 2}"
            n += 1
    return True, n, ("ok" if n else "kosong -- tidak ada rekaman pada rentang ini")


def unduh_partisi_meta(nama, y0, y1, percobaan=None):
    final = LAKE_META / f"meta_{nama}.csv"
    if final.exists() and final.stat().st_size > 0:
        return 0, True

    sementara = LAKE_META / f"meta_{nama}.part"
    fq = [f"CreateDate_dt:[{y0}-01-01T00:00:00Z TO {y1}-01-01T00:00:00Z]"]
    percobaan = percobaan or CFG["http_retry"]

    for i in range(percobaan):
        try:
            sementara.unlink(missing_ok=True)
            r = ambil(url_vvsearch(fq), stream=True,
                      timeout=CFG["http_timeout"], percobaan=1)
            with open(sementara, "wb") as fo:
                for potong in r.iter_content(chunk_size=1 << 20):
                    if potong:
                        fo.write(potong)

            utuh, n_baris, alasan = csv_utuh(sementara)
            if not utuh:
                raise RuntimeError(f"hasil tidak utuh: {alasan}")

            sementara.replace(final)
            return n_baris, False

        except Exception as ex:
            sementara.unlink(missing_ok=True)
            tunggu = min(2 ** i, 60)
            print(f"      {nama}: percobaan {i+1}/{percobaan} gagal "
                  f"({type(ex).__name__}: {str(ex)[:60]}) -- tunggu {tunggu}s")
            if i < percobaan - 1:
                time.sleep(tunggu)

    raise RuntimeError(f"partisi {nama} gagal setelah {percobaan} percobaan")

In [8]:
# JALANKAN JALUR B
# Aman dijalankan ulang: partisi yang berkasnya sudah ada akan dilewati.
# Partisi tanpa rekaman tetap dihitung berhasil dan disimpan sebagai berkas

TAHUN_META = [(f"pre{TAHUN_AWAL}", 1900, TAHUN_AWAL)] + \
             [(str(y), y, y + 1) for y in range(TAHUN_AWAL, TAHUN_AKHIR + 2)]

with Tahap("jalur B -- unduh metadata terstruktur", "INGESTION"):
    total_meta, dilewati, kosong, gagal = 0, 0, [], []
    for i, (nama, y0, y1) in enumerate(TAHUN_META, 1):
        try:
            n, sudah_ada = unduh_partisi_meta(nama, y0, y1)
        except Exception as ex:
            gagal.append((nama, f"{type(ex).__name__}: {str(ex)[:70]}"))
            print(f"  [{i}/{len(TAHUN_META)}] {nama:>8}  GAGAL -- dilewati")
            continue
        if sudah_ada:
            dilewati += 1
            print(f"  [{i}/{len(TAHUN_META)}] {nama:>8}  sudah ada -- dilewati")
        elif n == 0:
            kosong.append(nama)
            print(f"  [{i}/{len(TAHUN_META)}] {nama:>8}  kosong (tidak ada rekaman)")
        else:
            total_meta += n
            print(f"  [{i}/{len(TAHUN_META)}] {nama:>8}  {n:>9,} baris")

    print(f"\n  Baris metadata baru : {total_meta:,}")
    print(f"  Dilewati            : {dilewati} partisi")
    print(f"  Kosong (sah)        : {len(kosong)} partisi -> {', '.join(kosong) if kosong else '-'}")
    print(f"  Ukuran zona         : {ukuran(LAKE_META)}")
    if gagal:
        print(f"\n  {len(gagal)} partisi GAGAL -- jalankan ulang sel ini untuk mencobanya lagi:")
        for nama, sebab in gagal:
            print(f"    {nama}: {sebab}")
    else:
        print("  Seluruh partisi berhasil.")


--------------------------------------------------------------------
[>] INGESTION | jalur B -- unduh metadata terstruktur
  [1/49]  pre1980  sudah ada -- dilewati
  [2/49]     1980  sudah ada -- dilewati
  [3/49]     1981  sudah ada -- dilewati
  [4/49]     1982  sudah ada -- dilewati
  [5/49]     1983  sudah ada -- dilewati
  [6/49]     1984  sudah ada -- dilewati
  [7/49]     1985  sudah ada -- dilewati
  [8/49]     1986  sudah ada -- dilewati
  [9/49]     1987  sudah ada -- dilewati
  [10/49]     1988  sudah ada -- dilewati
  [11/49]     1989  sudah ada -- dilewati
  [12/49]     1990  sudah ada -- dilewati
  [13/49]     1991  sudah ada -- dilewati
  [14/49]     1992  sudah ada -- dilewati
  [15/49]     1993  sudah ada -- dilewati
  [16/49]     1994  sudah ada -- dilewati
  [17/49]     1995  sudah ada -- dilewati
  [18/49]     1996  sudah ada -- dilewati
  [19/49]     1997  sudah ada -- dilewati
  [20/49]     1998  sudah ada -- dilewati
  [21/49]     1999  sudah ada -- dilewati
  [

---
## Verifikasi dan manifest

Tiga pemeriksaan sebelum menyerahkan hasil ke notebook 02: berapa yang benar-benar mendarat, seberapa besar irisan nomor aksesi antar kedua jalur, dan seberapa lengkap kolom label yang akan dipakai sebagai target klasifikasi.

In [9]:
import csv, collections

with Tahap("verifikasi hasil ingestion", "MONITORING"):
    # --- jalur A: hitung defline tanpa memuat isi ke memori ---
    n_fasta, berkas_a = 0, sorted(LAKE_FASTA.glob("*.fasta.gz"))
    for p in berkas_a:
        with gzip.open(p, "rt", encoding="utf-8", errors="replace") as f:
            n_fasta += sum(1 for baris in f if baris.startswith(">"))
    print(f"  Jalur A : {n_fasta:,} sekuens dalam {len(berkas_a)} berkas  ({ukuran(LAKE_FASTA)})")

    # --- jalur B: hitung baris dan kelengkapan kolom label ---
    n_meta, terisi = 0, collections.Counter()
    aksesi_b = set()
    berkas_b = sorted(LAKE_META.glob("*.csv"))
    for p in berkas_b:
        if p.stat().st_size == 0:
            continue
        with open(p, encoding="utf-8", errors="replace", newline="") as f:
            for baris in csv.DictReader(f):
                n_meta += 1
                aksesi_b.add((baris.get("AccVer_s") or "").split(".")[0])
                for k in ("Serotype_s", "Segment_s", "Host_s",
                          "CountryFull_s", "CollectionDate_s"):
                    if (baris.get(k) or "").strip():
                        terisi[k] += 1
    print(f"  Jalur B : {n_meta:,} baris dalam {len(berkas_b)} berkas  ({ukuran(LAKE_META)})")

    print("\n  Kelengkapan kolom jalur B:")
    for k in ("Serotype_s", "Segment_s", "Host_s", "CountryFull_s", "CollectionDate_s"):
        print(f"    {k:<18} {terisi[k]:>9,}  ({100*terisi[k]/max(n_meta,1):5.1f}%)")

    manifest = {
        "run_id": RUN_ID,
        "waktu": datetime.now(timezone.utc).isoformat(),
        "sumber": {"entrez_total": n_entrez, "datasets_total": n_datasets},
        "jalur_a": {"metode": "E-utilities esearch+efetch, FASTA",
                    "sekuens": n_fasta, "berkas": len(berkas_a),
                    "ukuran": ukuran(LAKE_FASTA)},
        "jalur_b": {"metode": "NCBI Virus vvsearch2, CSV terstruktur",
                    "baris": n_meta, "berkas": len(berkas_b),
                    "ukuran": ukuran(LAKE_META),
                    "kelengkapan": dict(terisi)},
    }
    (LAKE / "manifest_ingestion.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"\n  Manifest disimpan: {LAKE / 'manifest_ingestion.json'}")


--------------------------------------------------------------------
[>] MONITORING | verifikasi hasil ingestion
  Jalur A : 1,627,536 sekuens dalam 152 berkas  (162.2 MB)
  Jalur B : 1,644,153 baris dalam 49 berkas  (367.6 MB)

  Kelengkapan kolom jalur B:
    Serotype_s         1,592,813  ( 96.9%)
    Segment_s          1,590,448  ( 96.7%)
    Host_s             1,554,797  ( 94.6%)
    CountryFull_s      1,585,013  ( 96.4%)
    CollectionDate_s   1,567,973  ( 95.4%)

  Manifest disimpan: /workspace/lake/manifest_ingestion.json
[<] OK | 28.4 detik | RAM bebas 44.3 GB


In [10]:
display(ringkas_zona())
display(jejak_df())

print("\nSelesai. Lanjut ke 02_storage_quality.ipynb")
print("Notebook ini tidak memakai Spark, jadi tidak ada JVM yang perlu dimatikan.")

,zona,isi,ukuran
0,lake/fasta,152,162.2 MB
1,lake/meta_csv,49,367.6 MB
2,stage,1,1.3 KB
3,features,1,735.0 B
4,models,4,2.2 KB
5,graph,0,0.0 B
6,mart,0,0.0 B
7,output,3,237.4 KB


,run_id,lapisan,tahap,status,detik,ram_delta_gb,ram_bebas_gb,waktu
0,run_20260922T020600Z,DATA SOURCES,probe kedua sumber,OK,4.67,0.01,44.7,2026-09-22T02:06:05.442000+00:00
1,run_20260922T020600Z,DATA SOURCES,inventaris per tahun rilis,OK,47.20,0.18,44.6,2026-09-22T02:06:52.645719+00:00
2,run_20260922T020600Z,DATA QUALITY,bukti: [Segment] bukan field yang sah,OK,8.34,0.01,44.5,2026-09-22T02:07:00.987150+00:00
3,run_20260922T020600Z,INGESTION,jalur A — unduh FASTA seluruh arsip,OK,0.46,0.00,44.5,2026-09-22T02:07:01.463247+00:00
4,run_20260922T020600Z,INGESTION,jalur B -- unduh metadata terstruktur,OK,0.24,0.00,44.5,2026-09-22T02:07:01.721686+00:00
5,run_20260922T020600Z,MONITORING,verifikasi hasil ingestion,OK,28.35,0.20,44.3,2026-09-22T02:07:30.079871+00:00



Selesai. Lanjut ke 02_storage_quality.ipynb
Notebook ini tidak memakai Spark, jadi tidak ada JVM yang perlu dimatikan.
